# Slices distribution analysis

In [1]:
import os
# Set the working directory
os.chdir("/home/jupyter-lukj08@vse.cz/VSE_bachelor_thesis_lumbar_spine_degeneration_classification")
print(os.getcwd())


/home/jupyter-lukj08@vse.cz/VSE_bachelor_thesis_lumbar_spine_degeneration_classification


In [2]:
# Used packages 
import opendatasets as od
import pandas as pd
import shutil, json, zipfile, random, math
import warnings
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import pydicom
from pathlib import Path
from IPython.display import clear_output
from sklearn.model_selection import train_test_split

import cv2
from PIL import Image
import torchvision.transforms as transforms

import plotly.offline as pyo
from scipy.interpolate import griddata
import plotly.offline as pyo
from scipy.interpolate import griddata
import plotly.graph_objects as go
from tqdm.auto import tqdm

In [3]:
path = "./BC-data/data-rsna2024"
train_data = pd.read_csv("./BC-data/data-rsna2024/train.csv")
train_label = pd.read_csv("./BC-data/data-rsna2024/train_label_coordinates.csv")
train_des = pd.read_csv("./BC-data/data-rsna2024/train_series_descriptions.csv")
data_merged = pd.read_csv("./BC-data/data-rsna2024/data_merged.csv")
display(data_merged.head())

,row_id,study_id,series_id,condition,level,series_description,instance_number,x,y,severity,img_path,image_height,image_width,x_norm,y_norm
0,4003253_spinal_canal_stenosis_l1_l2,4003253,702807833,Spinal Canal Stenosis,L1/L2,Sagittal T2/STIR,8,322.831858,227.964602,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...,640,640,0.504425,0.356195
1,4003253_spinal_canal_stenosis_l2_l3,4003253,702807833,Spinal Canal Stenosis,L2/L3,Sagittal T2/STIR,8,320.571429,295.714286,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...,640,640,0.500893,0.462054
2,4003253_spinal_canal_stenosis_l3_l4,4003253,702807833,Spinal Canal Stenosis,L3/L4,Sagittal T2/STIR,8,323.030303,371.818182,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...,640,640,0.504735,0.580966
3,4003253_spinal_canal_stenosis_l4_l5,4003253,702807833,Spinal Canal Stenosis,L4/L5,Sagittal T2/STIR,8,335.292035,427.327434,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...,640,640,0.523894,0.667699
4,4003253_spinal_canal_stenosis_l5_s1,4003253,702807833,Spinal Canal Stenosis,L5/S1,Sagittal T2/STIR,8,353.415929,483.964602,Normal/Mild,./BC-data/data-rsna2024/train_images/4003253/7...,640,640,0.552212,0.756195


In [4]:
def infer_plane_from_description(desc):
    desc = str(desc).lower()
    if "sag" in desc:
        return "Sagittal"
    if "ax" in desc:
        return "Axial"
    if "cor" in desc:
        return "Coronal"
    return "Unknown"

data_merged = data_merged.copy()
data_merged["plane"] = data_merged["series_description"].apply(infer_plane_from_description)

data_merged[["series_description", "plane"]].drop_duplicates().sort_values("series_description")

,series_description,plane
15,Axial T2,Axial
5,Sagittal T1,Sagittal
0,Sagittal T2/STIR,Sagittal


In [5]:
def read_dicom_geometry(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True)

    ipp = np.array(ds.ImagePositionPatient, dtype=float)
    iop = np.array(ds.ImageOrientationPatient, dtype=float)

    row_dir = iop[:3]
    col_dir = iop[3:]
    normal = np.cross(row_dir, col_dir)

    slice_pos = float(np.dot(ipp, normal))

    return {
        "img_path": str(path),
        "instance_number": int(getattr(ds, "InstanceNumber", -1)),
        "slice_pos": slice_pos,

        "rows": int(ds.Rows),
        "cols": int(ds.Columns),
        "pixel_spacing_0": float(ds.PixelSpacing[0]),
        "pixel_spacing_1": float(ds.PixelSpacing[1]),

        "ipp_x": ipp[0],
        "ipp_y": ipp[1],
        "ipp_z": ipp[2],

        "row_x": row_dir[0],
        "row_y": row_dir[1],
        "row_z": row_dir[2],

        "col_x": col_dir[0],
        "col_y": col_dir[1],
        "col_z": col_dir[2],

        "normal_x": normal[0],
        "normal_y": normal[1],
        "normal_z": normal[2],
    }

In [6]:
def read_dicom_geometry(path):
    ds = pydicom.dcmread(path, stop_before_pixels=True)

    ipp = np.array(ds.ImagePositionPatient, dtype=float)
    iop = np.array(ds.ImageOrientationPatient, dtype=float)

    row_dir = iop[:3]
    col_dir = iop[3:]
    normal = np.cross(row_dir, col_dir)

    slice_pos = float(np.dot(ipp, normal))

    return {
        "img_path": str(path),
        "instance_number": int(getattr(ds, "InstanceNumber", -1)),
        "slice_pos": slice_pos,

        "rows": int(ds.Rows),
        "cols": int(ds.Columns),
        "pixel_spacing_0": float(ds.PixelSpacing[0]),
        "pixel_spacing_1": float(ds.PixelSpacing[1]),

        "ipp_x": ipp[0],
        "ipp_y": ipp[1],
        "ipp_z": ipp[2],

        "row_x": row_dir[0],
        "row_y": row_dir[1],
        "row_z": row_dir[2],

        "col_x": col_dir[0],
        "col_y": col_dir[1],
        "col_z": col_dir[2],

        "normal_x": normal[0],
        "normal_y": normal[1],
        "normal_z": normal[2],
    }

In [7]:
def get_series_dir_from_img_path(path):
    return Path(path).parent


def build_all_series_paths_from_annotated_data(data_merged):
    """
    For every annotated series, find all DICOM files in the same series folder.
    This recovers unannotated slices too.
    """
    series_dirs = (
        data_merged[["study_id", "series_id", "plane", "series_description", "img_path"]]
        .dropna(subset=["img_path"])
        .drop_duplicates(subset=["study_id", "series_id"])
        .copy()
    )

    series_dirs["series_dir"] = series_dirs["img_path"].apply(lambda p: str(get_series_dir_from_img_path(p)))

    all_rows = []

    for _, row in tqdm(series_dirs.iterrows(), total=len(series_dirs)):
        series_dir = Path(row["series_dir"])

        dicom_paths = sorted(series_dir.glob("*.dcm"))

        # fallback, in case files do not have .dcm suffix
        if len(dicom_paths) == 0:
            dicom_paths = sorted([p for p in series_dir.iterdir() if p.is_file()])

        for p in dicom_paths:
            all_rows.append({
                "study_id": row["study_id"],
                "series_id": row["series_id"],
                "plane": row["plane"],
                "series_description": row["series_description"],
                "img_path": str(p),
                "series_dir": str(series_dir),
            })

    return pd.DataFrame(all_rows)


all_series_paths = build_all_series_paths_from_annotated_data(data_merged)

print("Annotated image paths:", data_merged["img_path"].nunique())
print("Recovered all series image paths:", all_series_paths["img_path"].nunique())
display(all_series_paths.head())

  0%|          | 0/6291 [00:00<?, ?it/s]

Annotated image paths: 24544
Recovered all series image paths: 147116


,study_id,series_id,plane,series_description,img_path,series_dir
0,4003253,702807833,Sagittal,Sagittal T2/STIR,BC-data/data-rsna2024/train_images/4003253/702...,BC-data/data-rsna2024/train_images/4003253/702...
1,4003253,702807833,Sagittal,Sagittal T2/STIR,BC-data/data-rsna2024/train_images/4003253/702...,BC-data/data-rsna2024/train_images/4003253/702...
2,4003253,702807833,Sagittal,Sagittal T2/STIR,BC-data/data-rsna2024/train_images/4003253/702...,BC-data/data-rsna2024/train_images/4003253/702...
3,4003253,702807833,Sagittal,Sagittal T2/STIR,BC-data/data-rsna2024/train_images/4003253/702...,BC-data/data-rsna2024/train_images/4003253/702...
4,4003253,702807833,Sagittal,Sagittal T2/STIR,BC-data/data-rsna2024/train_images/4003253/702...,BC-data/data-rsna2024/train_images/4003253/702...


In [9]:
# Rebuild all_slices_geom if it does not exist yet
if "all_slices_geom" not in globals():
    all_slices_geom = all_series_paths.merge(
        all_geom,
        on="img_path",
        how="left"
    )

    all_slices_geom = add_true_series_slice_positions(all_slices_geom)

print(all_slices_geom.shape)
display(all_slices_geom.head())

NameError: name 'all_geom' is not defined

In [8]:
import pandas as pd

# All series that contain at least one annotation for each condition
condition_series = (
    data_merged
    .groupby("condition")[["study_id", "series_id"]]
    .apply(lambda x: x.drop_duplicates())
    .reset_index(level=0)
    .reset_index(drop=True)
)

# Number of all images in every corresponding series
series_image_counts = (
    all_slices_geom
    .groupby(["study_id", "series_id"])
    .agg(
        n_images_in_series=("img_path", "nunique"),
        plane=("plane", "first"),
        series_description=("series_description", "first"),
    )
    .reset_index()
)

condition_series = condition_series.merge(
    series_image_counts,
    on=["study_id", "series_id"],
    how="left",
)

# Number of annotated images per condition
# nunique img_path = unique annotated DICOM images
annotated_counts = (
    data_merged
    .groupby("condition")
    .agg(
        n_annotated_rows=("row_id", "size"),
        n_annotated_images=("img_path", "nunique"),
        n_annotated_studies=("study_id", "nunique"),
        n_annotated_series=("series_id", "nunique"),
    )
    .reset_index()
)

# Final condition-level summary
condition_summary = (
    condition_series
    .groupby("condition")
    .agg(
        n_studies=("study_id", "nunique"),
        n_series=("series_id", "nunique"),
        n_total_images_in_corresponding_series=("n_images_in_series", "sum"),
    )
    .reset_index()
    .merge(
        annotated_counts,
        on="condition",
        how="left",
    )
)

condition_summary["annotated_image_ratio"] = (
    condition_summary["n_annotated_images"]
    / condition_summary["n_total_images_in_corresponding_series"]
)

condition_summary["annotated_row_ratio"] = (
    condition_summary["n_annotated_rows"]
    / condition_summary["n_total_images_in_corresponding_series"]
)

display(condition_summary)

NameError: name 'all_slices_geom' is not defined

In [ ]:
def build_geometry_for_all_slices(all_series_paths):
    rows = []

    for path in tqdm(all_series_paths["img_path"].drop_duplicates().tolist()):
        try:
            rows.append(read_dicom_geometry(path))
        except Exception as e:
            rows.append({
                "img_path": str(path),
                "instance_number": np.nan,
                "slice_pos": np.nan,
                "rows": np.nan,
                "cols": np.nan,
                "pixel_spacing_0": np.nan,
                "pixel_spacing_1": np.nan,
                "ipp_x": np.nan,
                "ipp_y": np.nan,
                "ipp_z": np.nan,
                "row_x": np.nan,
                "row_y": np.nan,
                "row_z": np.nan,
                "col_x": np.nan,
                "col_y": np.nan,
                "col_z": np.nan,
                "normal_x": np.nan,
                "normal_y": np.nan,
                "normal_z": np.nan,
                "error": str(e),
            })

    return pd.DataFrame(rows)


all_geom = build_geometry_for_all_slices(all_series_paths)

all_slices_geom = all_series_paths.merge(
    all_geom,
    on="img_path",
    how="left"
)

display(all_slices_geom.head())

In [ ]:
def add_true_series_slice_positions(all_slices_geom):
    out = all_slices_geom.copy()

    group_cols = ["study_id", "series_id"]

    # Sort by physical slice position within each series
    out = out.sort_values(group_cols + ["slice_pos", "instance_number"]).reset_index(drop=True)

    out["slice_rank_true"] = (
        out.groupby(group_cols)
        .cumcount()
        + 1
    )

    out["n_slices_true"] = (
        out.groupby(group_cols)["img_path"]
        .transform("nunique")
    )

    out["rel_slice_true"] = np.where(
        out["n_slices_true"] > 1,
        (out["slice_rank_true"] - 1) / (out["n_slices_true"] - 1),
        0.5
    )

    out["rel_slice_centered_true"] = 2 * out["rel_slice_true"] - 1

    out["slices_before_true"] = out["slice_rank_true"] - 1
    out["slices_after_true"] = out["n_slices_true"] - out["slice_rank_true"]

    return out


all_slices_geom = add_true_series_slice_positions(all_slices_geom)

display(
    all_slices_geom[
        ["study_id", "series_id", "plane", "series_description", "img_path",
         "instance_number", "slice_pos", "slice_rank_true", "n_slices_true",
         "rel_slice_true", "slices_before_true", "slices_after_true"]
    ].head()
)

In [ ]:
import matplotlib.pyplot as plt

sagittal_series_n_slices = (
    all_slices_geom[all_slices_geom["plane"].eq("Sagittal")]
    [["study_id", "series_id", "series_description", "n_slices_true"]]
    .drop_duplicates(["study_id", "series_id"])
    .copy()
)

dist = (
    sagittal_series_n_slices["n_slices_true"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(10, 4))
dist.plot(kind="bar")

plt.title("Distribution of Number of Slices per Sagittal Series")
plt.xlabel("Number of slices in series (n_slices_true)")
plt.ylabel("Number of sagittal series")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
data_merged = data_merged.copy()
all_slices_geom = all_slices_geom.copy()

data_merged["img_path_key"] = data_merged["img_path"].astype(str).map(lambda p: str(Path(p)))
all_slices_geom["img_path_key"] = all_slices_geom["img_path"].astype(str).map(lambda p: str(Path(p)))

true_slice_cols = [
    "img_path_key",
    "instance_number",
    "slice_pos",
    "rows",
    "cols",
    "pixel_spacing_0",
    "pixel_spacing_1",
    "slice_rank_true",
    "n_slices_true",
    "rel_slice_true",
    "rel_slice_centered_true",
    "slices_before_true",
    "slices_after_true",
]

df = data_merged.merge(
    all_slices_geom[true_slice_cols],
    on="img_path_key",
    how="left"
)

df["slice_rank"] = df["slice_rank_true"]
df["n_slices_observed"] = df["n_slices_true"]
df["rel_slice"] = df["rel_slice_true"]
df["rel_slice_centered"] = df["rel_slice_centered_true"]

print("Rows with true geometry:", df["rel_slice_true"].notna().sum(), "/", len(df))

In [ ]:
import os

data_merged = data_merged.copy()
all_slices_geom = all_slices_geom.copy()

data_merged["img_path_key"] = (
    data_merged["img_path"]
    .astype(str)
    .apply(os.path.normpath)
)

all_slices_geom["img_path_key"] = (
    all_slices_geom["img_path"]
    .astype(str)
    .apply(os.path.normpath)
)


In [ ]:
# Make sure path keys are normalized in both tables
import os

def normalize_path_for_merge(p):
    p = os.path.normpath(str(p))
    p = p.replace("\\", "/")
    p = p.lstrip("./")
    return p

data_merged = data_merged.copy()
all_slices_geom = all_slices_geom.copy()

data_merged["img_path_key"] = data_merged["img_path"].apply(normalize_path_for_merge)
all_slices_geom["img_path_key"] = all_slices_geom["img_path"].apply(normalize_path_for_merge)

In [ ]:
# ------------------------------------------------------------
# 1. Build annotation flag per DICOM slice
# ------------------------------------------------------------

ann = data_merged.copy()

# make sure path key exists
ann["img_path_key"] = ann["img_path"].apply(normalize_path_for_merge)

def annotation_group(row):
    condition = str(row["condition"]).lower()
    row_id = str(row.get("row_id", "")).lower()

    if condition == "spinal canal stenosis":
        return "Spinal canal stenosis"

    if "foraminal" in condition or "foraminal" in row_id:
        if "left" in condition or "left" in row_id:
            return "Left foraminal narrowing"
        if "right" in condition or "right" in row_id:
            return "Right foraminal narrowing"
        return "Foraminal narrowing"

    return "Other annotation"


ann["annotation_group"] = ann.apply(annotation_group, axis=1)

# If one slice has multiple annotations, keep all unique groups as separate rows
ann_slice_groups = (
    ann[["img_path_key", "annotation_group"]]
    .drop_duplicates()
)

In [ ]:
# ------------------------------------------------------------
# 2. LEFT JOIN annotations onto ALL slices
# ------------------------------------------------------------

all_slices_plot_df = all_slices_geom.copy()
all_slices_plot_df["img_path_key"] = all_slices_plot_df["img_path"].apply(normalize_path_for_merge)

plot_df = all_slices_plot_df.merge(
    ann_slice_groups,
    on="img_path_key",
    how="left"
)

plot_df["annotation_group"] = plot_df["annotation_group"].fillna("No annotation")

In [ ]:
# ------------------------------------------------------------
# 3. Keep sagittal only
# ------------------------------------------------------------

sag_plot = plot_df[plot_df["plane"].eq("Sagittal")].copy()

sag_plot = sag_plot.dropna(subset=["n_slices_true", "rel_slice_true"]).copy()
sag_plot["n_slices_true"] = sag_plot["n_slices_true"].astype(int)

print("All sagittal slice rows:", len(sag_plot))
print(sag_plot["annotation_group"].value_counts())


## Barplot - abs distribution

In [ ]:
g = sns.FacetGrid(
    sag_plot,
    col="n_slices_true",
    col_wrap=4,
    height=3.2,
    sharex=False,
    sharey=False
)

g.map_dataframe(
    sns.histplot,
    x="slice_rank_true",
    hue="annotation_group",
    discrete=True,
    stat="count",
    multiple="stack",
    alpha=0.8,
    shrink=0.85
)

for ax, n_slices in zip(g.axes.flatten(), g.col_names):
    n_slices = int(n_slices)
    center_rank = (n_slices + 1) / 2

    ax.axvline(
        center_rank,
        color="black",
        linestyle="--",
        linewidth=1
    )

    ax.set_xlim(0.5, n_slices + 0.5)

    # show fewer x ticks for larger series
    if n_slices <= 10:
        step = 1
    elif n_slices <= 20:
        step = 2
    elif n_slices <= 40:
        step = 5
    else:
        step = 10

    ticks = list(range(1, n_slices + 1, step))
    if n_slices not in ticks:
        ticks.append(n_slices)

    ax.set_xticks(ticks)
    ax.tick_params(axis="x", labelsize=7)   # smaller x-axis numbers
    ax.tick_params(axis="y", labelsize=8)

g.set_axis_labels(
    "Slice rank within full DICOM series",
    "Absolute slice count"
)

g.set_titles("n_slices = {col_name}")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle(
    "Sagittal slice distribution: annotations vs no annotation\n"
    "one bar per true DICOM slice"
)

plt.show()

## KDE density distribution plot

In [ ]:
sag = df[df["plane"].eq("Sagittal")].copy()

# keep only annotated sagittal rows with true full-series position
sag = sag.dropna(subset=["rel_slice_true"]).copy()

plt.figure(figsize=(12, 5))

sns.kdeplot(
    data=sag,
    x="rel_slice_true",
    hue="condition",
    common_norm=False,
    fill=True,
    alpha=0.5,
    linewidth=2,
    bw_adjust=2
)

plt.axvline(0.5, color="black", linestyle="--", linewidth=1)

plt.title("Sagittal condition distribution across true relative slice position")
plt.xlabel("True relative slice position within full DICOM series: 0 = first slice, 1 = last slice")
plt.ylabel("Density")
plt.tight_layout()
plt.show()

In [ ]:
LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

KDE_BW = 1.5

sag = df[df["plane"].eq("Sagittal")].copy()
sag = sag.dropna(subset=["rel_slice_true", "level", "condition"]).copy()
sag["level"] = pd.Categorical(sag["level"], categories=LEVEL_ORDER, ordered=True)

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True, sharey=True)
axes = axes.flatten()

# First 5 plots = individual levels
for i, level in enumerate(LEVEL_ORDER):
    ax = axes[i]
    d = sag[sag["level"] == level]

    if len(d) == 0:
        ax.set_visible(False)
        continue

    sns.kdeplot(
        data=d,
        x="rel_slice_true",
        hue="condition",
        common_norm=False,
        fill=True,
        alpha=,
        linewidth=2,
        ax=ax,
        bw_adjust=KDE_BW
    )

    ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
    ax.set_title(level)
    ax.set_xlabel("True relative slice position")
    ax.set_ylabel("Density")

    # optional: remove per-axis legends
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

# 6th plot = all levels combined
ax = axes[5]

sns.kdeplot(
    data=sag,
    x="rel_slice_true",
    hue="condition",
    common_norm=False,
    fill=True,
    alpha=0.25,
    linewidth=2,
    ax=ax
)

ax.axvline(0.5, color="black", linestyle="--", linewidth=1)
ax.set_title("All levels")
ax.set_xlabel("True relative slice position")
ax.set_ylabel("Density")

# optional: keep only legend on the 6th plot
leg = ax.get_legend()
if leg is not None:
    leg.set_title("Condition")

plt.suptitle(
    "Sagittal condition distribution across true relative slice position\n(level-specific + all levels)",
    y=1.02
)
plt.tight_layout()
plt.show()

## Geometry distribution
-> all sagittal slices are equidistant 

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import pydicom
from tqdm.auto import tqdm


def normalize_path_key(path):
    return str(path).replace("\\", "/").replace("./", "")


def infer_plane_from_normal(normal):
    """
    Infer anatomical plane from the dominant direction of the slice normal.
    DICOM patient coordinates are LPS:
        x = left/right
        y = posterior/anterior
        z = superior/inferior
    """
    axis = int(np.argmax(np.abs(normal)))

    if axis == 0:
        return "Sagittal"
    elif axis == 1:
        return "Coronal"
    elif axis == 2:
        return "Axial"
    else:
        return "Unknown"


def read_dicom_geometry(dcm_path):
    dcm_path = Path(dcm_path)

    ds = pydicom.dcmread(
        str(dcm_path),
        stop_before_pixels=True,
        force=True,
    )

    ipp = np.array(ds.ImagePositionPatient, dtype=float)
    iop = np.array(ds.ImageOrientationPatient, dtype=float)

    row_cosine = iop[:3]
    col_cosine = iop[3:]

    normal = np.cross(row_cosine, col_cosine)
    normal = normal / (np.linalg.norm(normal) + 1e-8)

    slice_pos = float(np.dot(ipp, normal))

    plane_from_geometry = infer_plane_from_normal(normal)

    pixel_spacing = getattr(ds, "PixelSpacing", [np.nan, np.nan])

    return {
        "img_path": normalize_path_key(dcm_path),
        "img_path_key": normalize_path_key(dcm_path),

        "study_id": str(dcm_path.parts[-3]),
        "series_id": str(dcm_path.parts[-2]),

        "instance_number": int(getattr(ds, "InstanceNumber", -1)),
        "rows": int(getattr(ds, "Rows", -1)),
        "cols": int(getattr(ds, "Columns", -1)),

        "series_description": str(getattr(ds, "SeriesDescription", "")),

        "ipp_x": float(ipp[0]),
        "ipp_y": float(ipp[1]),
        "ipp_z": float(ipp[2]),

        "row_cos_x": float(row_cosine[0]),
        "row_cos_y": float(row_cosine[1]),
        "row_cos_z": float(row_cosine[2]),

        "col_cos_x": float(col_cosine[0]),
        "col_cos_y": float(col_cosine[1]),
        "col_cos_z": float(col_cosine[2]),

        "normal_x": float(normal[0]),
        "normal_y": float(normal[1]),
        "normal_z": float(normal[2]),

        "slice_pos": slice_pos,
        "plane_from_geometry": plane_from_geometry,

        "pixel_spacing_row": float(pixel_spacing[0]) if len(pixel_spacing) > 0 else np.nan,
        "pixel_spacing_col": float(pixel_spacing[1]) if len(pixel_spacing) > 1 else np.nan,

        "slice_thickness": float(getattr(ds, "SliceThickness", np.nan)),
        "spacing_between_slices": float(getattr(ds, "SpacingBetweenSlices", np.nan)),
    }


def build_geometry_table(image_root):
    image_root = Path(image_root)

    dcm_paths = sorted(image_root.glob("*/*/*.dcm"))

    print("DICOM files found:", len(dcm_paths))

    rows = []

    for path in tqdm(dcm_paths):
        try:
            rows.append(read_dicom_geometry(path))
        except Exception as e:
            rows.append({
                "img_path": normalize_path_key(path),
                "img_path_key": normalize_path_key(path),
                "study_id": str(path.parts[-3]),
                "series_id": str(path.parts[-2]),
                "read_error": str(e),
            })

    geom = pd.DataFrame(rows)

    return geom

In [ ]:
IMAGE_ROOT = Path("BC-data/data-rsna2024/train_images")

geometry_df = build_geometry_table(IMAGE_ROOT)

print(geometry_df.shape)
display(geometry_df.head())

In [ ]:
def add_geometry_ranks(geom):
    geom = geom.copy()

    geom["study_id"] = geom["study_id"].astype(str)
    geom["series_id"] = geom["series_id"].astype(str)

    geom = geom.dropna(
        subset=[
            "study_id",
            "series_id",
            "slice_pos",
            "ipp_x",
            "ipp_y",
            "ipp_z",
            "plane_from_geometry",
        ]
    ).copy()

    # Patient-axis coordinate based on inferred plane
    geom["patient_axis_pos"] = np.nan

    geom.loc[
        geom["plane_from_geometry"].eq("Sagittal"),
        "patient_axis_pos"
    ] = geom.loc[
        geom["plane_from_geometry"].eq("Sagittal"),
        "ipp_x"
    ]

    geom.loc[
        geom["plane_from_geometry"].eq("Coronal"),
        "patient_axis_pos"
    ] = geom.loc[
        geom["plane_from_geometry"].eq("Coronal"),
        "ipp_y"
    ]

    geom.loc[
        geom["plane_from_geometry"].eq("Axial"),
        "patient_axis_pos"
    ] = geom.loc[
        geom["plane_from_geometry"].eq("Axial"),
        "ipp_z"
    ]

    ranked_series = []

    for (study_id, series_id), d in geom.groupby(["study_id", "series_id"]):
        d = d.copy()

        # ------------------------------------------------------------
        # 1. Raw geometry ordering using DICOM plane normal
        # ------------------------------------------------------------
        d = d.sort_values("slice_pos").copy()
        d["slice_rank_geom"] = np.arange(1, len(d) + 1)
        d["n_slices_geom"] = len(d)

        if len(d) > 1:
            d["rel_slice_geom"] = (d["slice_rank_geom"] - 1) / (len(d) - 1)
            d["rel_slice_centered_geom"] = 2 * d["rel_slice_geom"] - 1
        else:
            d["rel_slice_geom"] = 0.5
            d["rel_slice_centered_geom"] = 0.0

        # ------------------------------------------------------------
        # 2. Standardized patient-axis ordering
        # ------------------------------------------------------------
        d = d.sort_values("patient_axis_pos").copy()
        d["slice_rank_patient_axis"] = np.arange(1, len(d) + 1)
        d["n_slices_patient_axis"] = len(d)

        if len(d) > 1:
            d["rel_slice_patient_axis"] = (
                d["slice_rank_patient_axis"] - 1
            ) / (len(d) - 1)

            d["rel_slice_centered_patient_axis"] = (
                2 * d["rel_slice_patient_axis"] - 1
            )
        else:
            d["rel_slice_patient_axis"] = 0.5
            d["rel_slice_centered_patient_axis"] = 0.0

        ranked_series.append(d)

    geom_ranked = pd.concat(ranked_series, ignore_index=True)

    geom_ranked = geom_ranked.sort_values(
        ["study_id", "series_id", "slice_rank_patient_axis"]
    ).reset_index(drop=True)

    return geom_ranked

In [ ]:
geometry_df = add_geometry_ranks(geometry_df)

display(
    geometry_df[
        [
            "study_id",
            "series_id",
            "series_description",
            "plane_from_geometry",
            "instance_number",
            "ipp_x",
            "ipp_y",
            "ipp_z",
            "slice_pos",
            "patient_axis_pos",
            "slice_rank_geom",
            "rel_slice_geom",
            "slice_rank_patient_axis",
            "rel_slice_patient_axis",
            "img_path_key",
        ]
    ].head(10)
)

In [ ]:
sag_geom = geometry_df[
    geometry_df["plane_from_geometry"].eq("Sagittal")
].copy()

def check_slice_spacing_equidistance(
    geometry_df,
    plane_col="plane_from_geometry",
    pos_col="slice_pos",
    atol_mm=0.05,
    rtol=0.01,
):
    geom = geometry_df.copy()

    geom = geom.dropna(
        subset=["study_id", "series_id", plane_col, pos_col]
    ).copy()

    rows = []

    for (study_id, series_id), d in geom.groupby(["study_id", "series_id"]):
        d = d.sort_values(pos_col).copy()

        positions = d[pos_col].astype(float).to_numpy()
        diffs = np.diff(positions)

        if len(diffs) == 0:
            rows.append({
                "study_id": study_id,
                "series_id": series_id,
                "plane": d[plane_col].iloc[0],
                "n_slices": len(d),
                "median_spacing_mm": np.nan,
                "min_spacing_mm": np.nan,
                "max_spacing_mm": np.nan,
                "std_spacing_mm": np.nan,
                "max_abs_dev_from_median_mm": np.nan,
                "is_equidistant": True,
            })
            continue

        abs_diffs = np.abs(diffs)
        median_spacing = np.median(abs_diffs)

        abs_dev = np.abs(abs_diffs - median_spacing)
        max_abs_dev = abs_dev.max()

        is_equidistant = np.allclose(
            abs_diffs,
            median_spacing,
            atol=atol_mm,
            rtol=rtol,
        )

        rows.append({
            "study_id": study_id,
            "series_id": series_id,
            "plane": d[plane_col].iloc[0],
            "series_description": d["series_description"].iloc[0] if "series_description" in d.columns else None,
            "n_slices": len(d),
            "median_spacing_mm": median_spacing,
            "min_spacing_mm": abs_diffs.min(),
            "max_spacing_mm": abs_diffs.max(),
            "std_spacing_mm": abs_diffs.std(),
            "max_abs_dev_from_median_mm": max_abs_dev,
            "is_equidistant": bool(is_equidistant),
        })

    return pd.DataFrame(rows)


spacing_report = check_slice_spacing_equidistance(
    sag_geom,
    plane_col="plane_from_geometry",
    pos_col="slice_pos",
    atol_mm=0.05,
    rtol=0.01,
)

print("Total series:", len(spacing_report))
print("Equidistant series:", spacing_report["is_equidistant"].sum())
print("Non-equidistant series:", (~spacing_report["is_equidistant"]).sum())

display(
    spacing_report
    .sort_values("max_abs_dev_from_median_mm", ascending=False)
    .head(10)
)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

sag_geom = geometry_df[
    geometry_df["plane_from_geometry"].eq("Sagittal")
].copy()

sag_geom = sag_geom.dropna(
    subset=["rel_slice_geom", "rel_slice_patient_axis"]
).copy()

# optional: sample if there are too many points
PLOT_SAMPLE = 20000

if len(sag_geom) > PLOT_SAMPLE:
    plot_df = sag_geom.sample(PLOT_SAMPLE, random_state=42).copy()
else:
    plot_df = sag_geom.copy()

plt.figure(figsize=(7, 4))

sns.scatterplot(
    data=plot_df,
    x="rel_slice_geom",
    y="rel_slice_patient_axis",
    alpha=0.25,
    s=20,
    edgecolor=None,
)

# identity line
plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1, color="black")

plt.title("Sagittal slices: geometry relative position vs patient-axis relative position")
plt.xlabel("rel_slice_geom")
plt.ylabel("rel_slice_patient_axis")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.grid(alpha=0.2)
plt.show()

## Rule-based Model for slice selection

In [ ]:
#display(data_merged.head(5))
#display(all_slices_geom.head(5))


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

TRAIN_SIZE = 0.70
VAL_SIZE = 0.15
TEST_SIZE = 0.15
RANDOM_STATE = 42

# Add true slice geometry to data_merged before splitting
geom_cols = [
    "img_path_key",
    "slice_rank_true",
    "n_slices_true",
    "rel_slice_true",
    "rel_slice_centered_true",
    "slices_before_true",
    "slices_after_true",
]

data_split = data_merged.merge(
    all_slices_geom[geom_cols].drop_duplicates("img_path_key"),
    on="img_path_key",
    how="left",
)

# First split: train vs temp
gss1 = GroupShuffleSplit(
    n_splits=1,
    train_size=TRAIN_SIZE,
    random_state=RANDOM_STATE,
)

train_idx, temp_idx = next(
    gss1.split(data_split, groups=data_split["study_id"])
)

train_df = data_split.iloc[train_idx].copy()
temp_df = data_split.iloc[temp_idx].copy()

# Second split: val vs test
val_relative_size = VAL_SIZE / (VAL_SIZE + TEST_SIZE)

gss2 = GroupShuffleSplit(
    n_splits=1,
    train_size=val_relative_size,
    random_state=RANDOM_STATE,
)

val_idx, test_idx = next(
    gss2.split(temp_df, groups=temp_df["study_id"])
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

print("Rows:")
print("train:", train_df.shape)
print("val:  ", val_df.shape)
print("test: ", test_df.shape)

print("\nUnique studies:")
print("train:", train_df["study_id"].nunique())
print("val:  ", val_df["study_id"].nunique())
print("test: ", test_df["study_id"].nunique())

print("\nMissing slice geometry:")
print("train:", train_df["slice_rank_true"].isna().sum())
print("val:  ", val_df["slice_rank_true"].isna().sum())
print("test: ", test_df["slice_rank_true"].isna().sum())

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

sagittal_slices = all_slices_geom[
    all_slices_geom["plane"].eq("Sagittal")
].copy()

sagittal_slices = sagittal_slices.dropna(
    subset=[
        "study_id",
        "series_id",
        "img_path",
        "slice_rank_true",
        "n_slices_true",
        "rel_slice_true",
    ]
).copy()

sagittal_slices["study_id"] = sagittal_slices["study_id"].astype(str)
sagittal_slices["series_id"] = sagittal_slices["series_id"].astype(str)
sagittal_slices["slice_rank_true"] = sagittal_slices["slice_rank_true"].astype(int)
sagittal_slices["n_slices_true"] = sagittal_slices["n_slices_true"].astype(int)

sagittal_slices = sagittal_slices.sort_values(
    ["study_id", "series_id", "slice_rank_true"]
).reset_index(drop=True)

print("Sagittal slice rows:", sagittal_slices.shape)
print("Sagittal series:", sagittal_slices[["study_id", "series_id"]].drop_duplicates().shape[0])

In [ ]:
LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

RELATIVE_SLICE_RULES = {
    "spinal_canal": {
        "all": 0.50,
        "L1/L2": 0.50,
        "L2/L3": 0.50,
        "L3/L4": 0.50,
        "L4/L5": 0.50,
        "L5/S1": 0.50,
    },
    "left_foraminal": {
        "all": 0.29,
        "L1/L2": 0.30,
        "L2/L3": 0.30,
        "L3/L4": 0.29,
        "L4/L5": 0.28,
        "L5/S1": 0.27,
    },
    "right_foraminal": {
        "all": 0.76,
        "L1/L2": 0.73,
        "L2/L3": 0.74,
        "L3/L4": 0.76,
        "L4/L5": 0.76,
        "L5/S1": 0.78,
    },
}

In [ ]:
def normalize_condition_for_slice_selection(row):
    """
    Maps RSNA condition + row_id to sagittal slice selector task.
    """
    condition = str(row.get("condition", "")).lower()
    row_id = str(row.get("row_id", "")).lower()
    text = condition + " " + row_id

    if "spinal canal stenosis" in text:
        return "spinal_canal"

    if "foraminal" in text:
        if "left" in text:
            return "left_foraminal"
        if "right" in text:
            return "right_foraminal"

    return None


def get_level_key(row):
    level = row.get("level", None)

    if pd.isna(level):
        return "all"

    level = str(level)

    if level in LEVEL_ORDER:
        return level

    return "all"


def relative_position_to_slice_rank(rel_pos, n_slices):
    """
    Converts relative position [0, 1] to 1-based slice_rank_true.
    """
    rel_pos = float(np.clip(rel_pos, 0.0, 1.0))

    if n_slices <= 1:
        return 1

    idx_0 = int(round(rel_pos * (n_slices - 1)))
    rank_1 = idx_0 + 1

    return int(np.clip(rank_1, 1, n_slices))

def add_triplet_interval_columns(df):
    df = df.copy()

    df["selected_min_rank"] = df["selected_triplet_ranks"].apply(min)
    df["selected_max_rank"] = df["selected_triplet_ranks"].apply(max)

    df["selected_min_rel"] = (df["selected_min_rank"] - 1) / (df["selected_n_slices"] - 1)
    df["selected_max_rel"] = (df["selected_max_rank"] - 1) / (df["selected_n_slices"] - 1)

    df["overall_coverage_overlay"] = (
        (df["rel_slice_true"] >= df["selected_min_rel"])
        & (df["rel_slice_true"] <= df["selected_max_rel"])
    )

    return df

def make_triplet_ranks(center_rank, n_slices):
    """
    Always returns exactly 3 ranks: [center-1, center, center+1].
    Boundary ranks are clipped, so first/last slice may repeat.
    """
    return [
        int(np.clip(center_rank - 1, 1, n_slices)),
        int(np.clip(center_rank, 1, n_slices)),
        int(np.clip(center_rank + 1, 1, n_slices)),
    ]

In [ ]:
def add_rule_based_sagittal_triplets(
    df_split,
    sagittal_slices,
    relative_center_rules,
):
    """
    Adds one selected 2.5D sagittal triplet to each sagittal row.

    Output columns added:
        selector_task
        selector_level_key
        rule_rel_pos
        selected_center_rank
        selected_triplet_ranks
        selected_img_paths
        selected_img_path_prev
        selected_img_path_center
        selected_img_path_next
        selected_n_slices
    """

    df = df_split.copy()

    # Make IDs comparable with all_slices_geom
    df["study_id"] = df["study_id"].astype(str)
    df["series_id"] = df["series_id"].astype(str)

    # Keep only sagittal rows.
    # This naturally removes axial subarticular rows.
    if "plane" in df.columns:
        df = df[df["plane"].eq("Sagittal")].copy()

    selected_rows = []

    for _, row in df.iterrows():
        selector_task = normalize_condition_for_slice_selection(row)

        if selector_task is None:
            continue

        study_id = row["study_id"]
        series_id = row["series_id"]
        level_key = get_level_key(row)

        series_slices = sagittal_slices[
            sagittal_slices["study_id"].eq(study_id)
            & sagittal_slices["series_id"].eq(series_id)
        ].copy()

        if len(series_slices) == 0:
            continue

        series_slices = series_slices.sort_values("slice_rank_true").copy()

        n_slices = int(series_slices["n_slices_true"].iloc[0])

        task_rules = relative_center_rules.get(selector_task, {})
        rel_pos = task_rules.get(level_key, task_rules.get("all", 0.50))

        center_rank = relative_position_to_slice_rank(
            rel_pos=rel_pos,
            n_slices=n_slices,
        )

        triplet_ranks = make_triplet_ranks(
            center_rank=center_rank,
            n_slices=n_slices,
        )

        rank_to_path = dict(
            zip(
                series_slices["slice_rank_true"].astype(int),
                series_slices["img_path"],
            )
        )

        triplet_paths = [rank_to_path.get(rank) for rank in triplet_ranks]

        out = row.to_dict()

        out.update(
            {
                "selector_task": selector_task,
                "selector_level_key": level_key,
                "rule_rel_pos": float(rel_pos),
                "selected_center_rank": int(center_rank),
                "selected_triplet_ranks": triplet_ranks,
                "selected_img_paths": triplet_paths,
                "selected_img_path_prev": triplet_paths[0],
                "selected_img_path_center": triplet_paths[1],
                "selected_img_path_next": triplet_paths[2],
                "selected_n_slices": int(n_slices),
            }
        )

        selected_rows.append(out)

    return pd.DataFrame(selected_rows)

In [ ]:
train_selected_df = add_rule_based_sagittal_triplets(
    df_split=train_df,
    sagittal_slices=sagittal_slices,
    relative_center_rules=RELATIVE_SLICE_RULES,
)

val_selected_df = add_rule_based_sagittal_triplets(
    df_split=val_df,
    sagittal_slices=sagittal_slices,
    relative_center_rules=RELATIVE_SLICE_RULES,
)

test_selected_df = add_rule_based_sagittal_triplets(
    df_split=test_df,
    sagittal_slices=sagittal_slices,
    relative_center_rules=RELATIVE_SLICE_RULES,
)


display(
    train_selected_df[
        [
            "study_id",
            "series_id",
            "condition",
            "level",
            "selector_task",
            "rule_rel_pos",
            "selected_center_rank",
            "selected_triplet_ranks",
            "selected_img_path_prev",
            "selected_img_path_center",
            "selected_img_path_next",
        ]
    ].head()
)

In [ ]:
def evaluate_triplet_coverage(selected_df):
    df = selected_df.copy()

    if "slice_rank_true" not in df.columns:
        raise ValueError(
            "slice_rank_true is missing. "
            "Merge data_merged with all_slices_geom on img_path_key before splitting."
        )

    df = df.dropna(subset=["slice_rank_true"]).copy()
    df["slice_rank_true"] = df["slice_rank_true"].astype(int)

    df["triplet_contains_true_slice"] = df.apply(
        lambda row: row["slice_rank_true"] in row["selected_triplet_ranks"],
        axis=1,
    )

    df["abs_center_error_slices"] = (
        df["selected_center_rank"] - df["slice_rank_true"]
    ).abs()

    summary = (
        df.groupby(["selector_task", "level"], dropna=False)
        .agg(
            n=("triplet_contains_true_slice", "size"),
            coverage=("triplet_contains_true_slice", "mean"),
            mean_abs_center_error=("abs_center_error_slices", "mean"),
            median_abs_center_error=("abs_center_error_slices", "median"),
        )
        .reset_index()
        .sort_values(["selector_task", "level"])
    )

    return df, summary

In [ ]:
train_triplet_eval_df, train_triplet_summary = evaluate_triplet_coverage(train_selected_df)
val_triplet_eval_df, val_triplet_summary = evaluate_triplet_coverage(val_selected_df)
test_triplet_eval_df, test_triplet_summary = evaluate_triplet_coverage(test_selected_df)


print("Test triplet coverage:")
display(test_triplet_summary)

In [ ]:
def add_annotated_slice_coverage(selected_df):
    df = selected_df.copy()

    df["slice_rank_true"] = df["slice_rank_true"].astype(int)

    df["annotated_slice_covered"] = df.apply(
        lambda row: row["slice_rank_true"] in row["selected_triplet_ranks"],
        axis=1,
    )

    df["center_abs_error_slices"] = (
        df["selected_center_rank"] - df["slice_rank_true"]
    ).abs()

    return df


train_selected_df = add_annotated_slice_coverage(train_selected_df)
val_selected_df = add_annotated_slice_coverage(val_selected_df)
test_selected_df = add_annotated_slice_coverage(test_selected_df)

In [ ]:
def report_annotated_slice_coverage(df, name):
    print("=" * 80)
    print(name)
    print("=" * 80)

    print("\nOverall annotated slice coverage:")
    print(df["annotated_slice_covered"].mean())

    print("\nBy condition:")
    display(
        df.groupby("condition")
        .agg(
            n=("annotated_slice_covered", "size"),
            coverage=("annotated_slice_covered", "mean"),
            mean_center_abs_error=("center_abs_error_slices", "mean"),
            median_center_abs_error=("center_abs_error_slices", "median"),
        )
        .reset_index()
    )


report_annotated_slice_coverage(train_selected_df, "TRAIN")
report_annotated_slice_coverage(val_selected_df, "VALIDATION")
report_annotated_slice_coverage(test_selected_df, "TEST")

## KDE Model

In [ ]:
from scipy.stats import gaussian_kde

In [ ]:
LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

def normalize_condition_for_slice_selection(row):
    condition = str(row.get("condition", "")).lower()
    row_id = str(row.get("row_id", "")).lower()
    text = condition + " " + row_id

    if "spinal canal stenosis" in text:
        return "spinal_canal"

    if "foraminal" in text:
        if "left" in text:
            return "left_foraminal"
        if "right" in text:
            return "right_foraminal"

    return None


def get_level_key(row):
    level = row.get("level", None)

    if pd.isna(level):
        return "all"

    level = str(level)

    if level in LEVEL_ORDER:
        return level

    return "all"

In [ ]:
def fit_kde_relative_slice_centers(
    train_df,
    min_samples=20,
    bandwidth_adjust=1.25,
    grid_size=1001,
):
    """
    Fits KDE on train annotations and returns relative center rules.

    Output:
        kde_center_rules[selector_task][level] = best_rel_position
    """

    df = train_df.copy()

    if "rel_slice_true" not in df.columns:
        raise ValueError("rel_slice_true missing. Use data_split with geometry before splitting.")

    df["selector_task"] = df.apply(normalize_condition_for_slice_selection, axis=1)
    df["selector_level_key"] = df.apply(get_level_key, axis=1)

    df = df.dropna(subset=["selector_task", "rel_slice_true"]).copy()
    df = df[df["plane"].eq("Sagittal")].copy()

    x_grid = np.linspace(0.0, 1.0, grid_size)

    kde_center_rules = {}
    kde_diagnostics = []

    for task in sorted(df["selector_task"].dropna().unique()):
        kde_center_rules[task] = {}

        task_df = df[df["selector_task"].eq(task)].copy()

        # First fit global fallback for this task
        task_values = task_df["rel_slice_true"].dropna().astype(float).values

        if len(task_values) >= 2:
            kde = gaussian_kde(task_values)
            kde.set_bandwidth(kde.factor * bandwidth_adjust)

            density = kde(x_grid)
            best_rel = float(x_grid[np.argmax(density)])
        else:
            best_rel = float(np.median(task_values)) if len(task_values) else 0.50

        kde_center_rules[task]["all"] = best_rel

        kde_diagnostics.append(
            {
                "selector_task": task,
                "level": "all",
                "n": len(task_values),
                "kde_center_rel": best_rel,
                "used_fallback": False,
            }
        )

        # Then fit level-specific centers
        for level in LEVEL_ORDER:
            d = task_df[task_df["selector_level_key"].eq(level)].copy()
            values = d["rel_slice_true"].dropna().astype(float).values

            if len(values) >= min_samples and len(np.unique(values)) >= 2:
                kde = gaussian_kde(values)
                kde.set_bandwidth(kde.factor * bandwidth_adjust)

                density = kde(x_grid)
                best_rel = float(x_grid[np.argmax(density)])
                used_fallback = False
            else:
                best_rel = kde_center_rules[task]["all"]
                used_fallback = True

            kde_center_rules[task][level] = best_rel

            kde_diagnostics.append(
                {
                    "selector_task": task,
                    "level": level,
                    "n": len(values),
                    "kde_center_rel": best_rel,
                    "used_fallback": used_fallback,
                }
            )

    kde_diagnostics_df = pd.DataFrame(kde_diagnostics)

    return kde_center_rules, kde_diagnostics_df

In [ ]:
KDE_CENTER_RULES, kde_diagnostics_df = fit_kde_relative_slice_centers(
    train_df=train_df,
    min_samples=20,
    bandwidth_adjust=1.5,
    grid_size=1001,
)

print(KDE_CENTER_RULES)

In [ ]:
train_kde_selected_df = add_rule_based_sagittal_triplets(
    df_split=train_df,
    sagittal_slices=sagittal_slices,
    relative_center_rules=KDE_CENTER_RULES,
)

val_kde_selected_df = add_rule_based_sagittal_triplets(
    df_split=val_df,
    sagittal_slices=sagittal_slices,
    relative_center_rules=KDE_CENTER_RULES,
)

test_kde_selected_df = add_rule_based_sagittal_triplets(
    df_split=test_df,
    sagittal_slices=sagittal_slices,
    relative_center_rules=KDE_CENTER_RULES,
)

In [ ]:
train_kde_selected_df = add_annotated_slice_coverage(train_kde_selected_df)
val_kde_selected_df = add_annotated_slice_coverage(val_kde_selected_df)
test_kde_selected_df = add_annotated_slice_coverage(test_kde_selected_df)

train_kde_selected_df = add_triplet_interval_columns(train_kde_selected_df)
val_kde_selected_df = add_triplet_interval_columns(val_kde_selected_df)
test_kde_selected_df = add_triplet_interval_columns(test_kde_selected_df)

In [ ]:
report_annotated_slice_coverage(train_kde_selected_df, "TRAIN KDE")
report_annotated_slice_coverage(val_kde_selected_df, "VALIDATION KDE")
report_annotated_slice_coverage(test_kde_selected_df, "TEST KDE")


## Number of slices specific KDE
-> not any improvement 

In [ ]:
def get_n_slices_bin(n_slices):
    n_slices = int(n_slices)

    if n_slices <= 12:
        return "n_<=12"
    elif n_slices <= 16:
        return "n_13_16"
    elif n_slices <= 20:
        return "n_17_20"
    else:
        return "n_21_plus"

In [ ]:
from scipy.stats import gaussian_kde
import numpy as np
import pandas as pd

def fit_kde_center_rules_by_n_slices(
    train_df,
    min_samples=20,
    bandwidth_adjust=1.25,
    grid_size=1001,
):
    df = train_df.copy()

    df["selector_task"] = df.apply(normalize_condition_for_slice_selection, axis=1)
    df["selector_level_key"] = df.apply(get_level_key, axis=1)
    df["n_slices_bin"] = df["n_slices_true"].apply(get_n_slices_bin)

    df = df[
        df["plane"].eq("Sagittal")
        & df["selector_task"].notna()
        & df["rel_slice_true"].notna()
        & df["n_slices_true"].notna()
    ].copy()

    x_grid = np.linspace(0, 1, grid_size)

    kde_rules = {}
    rows = []

    def fit_center(values):
        values = np.asarray(values, dtype=float)

        if len(values) >= min_samples and len(np.unique(values)) >= 2:
            kde = gaussian_kde(values)
            kde.set_bandwidth(kde.factor * bandwidth_adjust)
            density = kde(x_grid)
            return float(x_grid[np.argmax(density)]), False

        if len(values) > 0:
            return float(np.median(values)), True

        return 0.50, True

    # Global fallback: task only
    global_task_centers = {}

    for task, task_df in df.groupby("selector_task"):
        center, used_fallback = fit_center(task_df["rel_slice_true"].values)
        global_task_centers[task] = center

    # Main rules: task × level × n_slices_bin
    for task, task_df in df.groupby("selector_task"):
        kde_rules[task] = {}

        for level in ["all"] + LEVEL_ORDER:
            kde_rules[task][level] = {}

            if level == "all":
                level_df = task_df
            else:
                level_df = task_df[task_df["selector_level_key"].eq(level)]

            # Fallback for task × level
            level_center, _ = fit_center(level_df["rel_slice_true"].values)

            if len(level_df) < min_samples:
                level_center = global_task_centers[task]

            for n_bin in ["n_<=12", "n_13_16", "n_17_20", "n_21_plus"]:
                bin_df = level_df[level_df["n_slices_bin"].eq(n_bin)]
                values = bin_df["rel_slice_true"].values

                if len(values) >= min_samples and len(np.unique(values)) >= 2:
                    center, used_fallback = fit_center(values)
                else:
                    center = level_center
                    used_fallback = True

                kde_rules[task][level][n_bin] = center

                rows.append(
                    {
                        "selector_task": task,
                        "level": level,
                        "n_slices_bin": n_bin,
                        "n": len(values),
                        "kde_rel_pos": center,
                        "used_fallback": used_fallback,
                    }
                )

    diagnostics_df = pd.DataFrame(rows)

    return kde_rules, diagnostics_df

In [ ]:
KDE_N_CENTER_RULES, kde_n_diagnostics_df = fit_kde_center_rules_by_n_slices(
    train_df=train_df,
    min_samples=20,
    bandwidth_adjust=1.25,
)

display(kde_n_diagnostics_df)

In [ ]:
def add_kde_n_slices_sagittal_triplets(
    df_split,
    sagittal_slices,
    kde_n_center_rules,
):
    df = df_split.copy()

    df["study_id"] = df["study_id"].astype(str)
    df["series_id"] = df["series_id"].astype(str)

    if "plane" in df.columns:
        df = df[df["plane"].eq("Sagittal")].copy()

    selected_rows = []

    for _, row in df.iterrows():
        selector_task = normalize_condition_for_slice_selection(row)

        if selector_task is None:
            continue

        study_id = row["study_id"]
        series_id = row["series_id"]
        level_key = get_level_key(row)

        series_slices = sagittal_slices[
            sagittal_slices["study_id"].eq(study_id)
            & sagittal_slices["series_id"].eq(series_id)
        ].copy()

        if len(series_slices) == 0:
            continue

        series_slices = series_slices.sort_values("slice_rank_true").copy()
        n_slices = int(series_slices["n_slices_true"].iloc[0])
        n_bin = get_n_slices_bin(n_slices)

        task_rules = kde_n_center_rules.get(selector_task, {})
        level_rules = task_rules.get(level_key, task_rules.get("all", {}))

        rel_pos = level_rules.get(n_bin, 0.50)

        center_rank = relative_position_to_slice_rank(
            rel_pos=rel_pos,
            n_slices=n_slices,
        )

        triplet_ranks = make_triplet_ranks(
            center_rank=center_rank,
            n_slices=n_slices,
        )

        rank_to_path = dict(
            zip(
                series_slices["slice_rank_true"].astype(int),
                series_slices["img_path"],
            )
        )

        triplet_paths = [rank_to_path.get(rank) for rank in triplet_ranks]

        out = row.to_dict()

        out.update(
            {
                "selector_method": "kde_n_slices",
                "selector_task": selector_task,
                "selector_level_key": level_key,
                "n_slices_bin": n_bin,
                "kde_rel_pos": float(rel_pos),
                "selected_center_rank": int(center_rank),
                "selected_triplet_ranks": triplet_ranks,
                "selected_img_paths": triplet_paths,
                "selected_img_path_prev": triplet_paths[0],
                "selected_img_path_center": triplet_paths[1],
                "selected_img_path_next": triplet_paths[2],
                "selected_n_slices": int(n_slices),
            }
        )

        selected_rows.append(out)

    return pd.DataFrame(selected_rows)

In [ ]:
train_kde_n_selected_df = add_kde_n_slices_sagittal_triplets(
    df_split=train_df,
    sagittal_slices=sagittal_slices,
    kde_n_center_rules=KDE_N_CENTER_RULES,
)

val_kde_n_selected_df = add_kde_n_slices_sagittal_triplets(
    df_split=val_df,
    sagittal_slices=sagittal_slices,
    kde_n_center_rules=KDE_N_CENTER_RULES,
)

test_kde_n_selected_df = add_kde_n_slices_sagittal_triplets(
    df_split=test_df,
    sagittal_slices=sagittal_slices,
    kde_n_center_rules=KDE_N_CENTER_RULES,
)

In [ ]:
train_kde_n_selected_df = add_annotated_slice_coverage(train_kde_n_selected_df)
val_kde_n_selected_df = add_annotated_slice_coverage(val_kde_n_selected_df)
test_kde_n_selected_df = add_annotated_slice_coverage(test_kde_n_selected_df)

train_kde_n_selected_df = add_triplet_interval_columns(train_kde_n_selected_df)
val_kde_n_selected_df = add_triplet_interval_columns(val_kde_n_selected_df)
test_kde_n_selected_df = add_triplet_interval_columns(test_kde_n_selected_df)

In [ ]:
report_annotated_slice_coverage(train_kde_n_selected_df, "TRAIN KDE + N SLICES")
report_annotated_slice_coverage(val_kde_n_selected_df, "VALIDATION KDE + N SLICES")
report_annotated_slice_coverage(test_kde_n_selected_df, "TEST KDE + N SLICES")

## Axial slices distribution

In [ ]:
import matplotlib.pyplot as plt

axial_series_n_slices = (
    all_slices_geom[all_slices_geom["plane"].eq("Axial")]
    [["study_id", "series_id", "series_description", "n_slices_true"]]
    .drop_duplicates(["study_id", "series_id"])
    .copy()
)

dist = (
    axial_series_n_slices["n_slices_true"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(12, 4))
dist.plot(kind="bar")

plt.title("Distribution of Number of Slices per Axial Series")
plt.xlabel("Number of slices in series (n_slices_true)")
plt.ylabel("Number of axial series")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

geom_cols = [
    "img_path_key",
    "ipp_x", "ipp_y", "ipp_z",
    "row_x", "row_y", "row_z",
    "col_x", "col_y", "col_z",
    "normal_x", "normal_y", "normal_z",
    "pixel_spacing_0", "pixel_spacing_1",
    "slice_rank_true",
    "series_id",
    "study_id",
    "plane",
]

ann_df = data_merged.merge(
    all_slices_geom[geom_cols].drop_duplicates("img_path_key"),
    on="img_path_key",
    how="left",
    suffixes=("", "_geom"),
)

# keep original columns if present, otherwise fill from geom
for col in ["study_id", "series_id", "plane"]:
    geom_col = f"{col}_geom"
    if geom_col in ann_df.columns:
        ann_df[col] = ann_df[col].fillna(ann_df[geom_col])
        ann_df = ann_df.drop(columns=[geom_col])

display(ann_df.head())

In [ ]:
def pixel_to_patient_3d(row):
    origin = np.array([row["ipp_x"], row["ipp_y"], row["ipp_z"]], dtype=float)

    row_dir = np.array([row["row_x"], row["row_y"], row["row_z"]], dtype=float)
    col_dir = np.array([row["col_x"], row["col_y"], row["col_z"]], dtype=float)

    ps_row = float(row["pixel_spacing_0"])  # spacing between image rows
    ps_col = float(row["pixel_spacing_1"])  # spacing between image columns

    # Correct DICOM mapping:
    # x = column coordinate -> row_dir, pixel_spacing_1
    # y = row coordinate    -> col_dir, pixel_spacing_0
    point = (
        origin
        + row["x"] * ps_col * row_dir
        + row["y"] * ps_row * col_dir
    )

    return point

In [ ]:
sag_ref = ann_df[
    ann_df["plane"].eq("Sagittal")
    & ann_df["condition"].eq("Spinal Canal Stenosis")
    & ann_df["level"].isin(LEVEL_ORDER)
].copy()

sag_points = sag_ref.apply(pixel_to_patient_3d, axis=1)
sag_ref["pt_x"] = sag_points.apply(lambda p: p[0])
sag_ref["pt_y"] = sag_points.apply(lambda p: p[1])
sag_ref["pt_z"] = sag_points.apply(lambda p: p[2])

sag_ref = (
    sag_ref[
        ["study_id", "level", "pt_x", "pt_y", "pt_z"]
    ]
    .dropna()
    .drop_duplicates(["study_id", "level"])
)

display(sag_ref.head())



In [ ]:
axial_ann = ann_df[
    ann_df["plane"].eq("Axial")
    & ann_df["level"].isin(LEVEL_ORDER)
].copy()

axial_ann = axial_ann.merge(
    sag_ref,
    on=["study_id", "level"],
    how="inner",
)

display(axial_ann.head())

In [ ]:
def point_to_plane_signed_distance(point_xyz, plane_origin_xyz, plane_normal_xyz):
    n = np.asarray(plane_normal_xyz, dtype=float)
    n_norm = np.linalg.norm(n)
    if n_norm == 0:
        return np.nan
    n = n / n_norm

    p = np.asarray(point_xyz, dtype=float)
    o = np.asarray(plane_origin_xyz, dtype=float)

    return float(np.dot(p - o, n))


def compute_plane_distance(row):
    point_xyz = [row["pt_x"], row["pt_y"], row["pt_z"]]
    plane_origin_xyz = [row["ipp_x"], row["ipp_y"], row["ipp_z"]]
    plane_normal_xyz = [row["normal_x"], row["normal_y"], row["normal_z"]]

    return point_to_plane_signed_distance(
        point_xyz=point_xyz,
        plane_origin_xyz=plane_origin_xyz,
        plane_normal_xyz=plane_normal_xyz,
    )


axial_ann["signed_plane_distance_mm"] = axial_ann.apply(compute_plane_distance, axis=1)
axial_ann["abs_plane_distance_mm"] = axial_ann["signed_plane_distance_mm"].abs()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

def unit_normal(row):
    n = np.array(
        [row["normal_x"], row["normal_y"], row["normal_z"]],
        dtype=float,
    )
    norm = np.linalg.norm(n)
    return n / norm if norm != 0 else np.array([np.nan, np.nan, np.nan])


def point_dot_normal(px, py, pz, nx, ny, nz):
    return px * nx + py * ny + pz * nz


plot_df = axial_ann.copy()

# normalize axial slice normal
normals = plot_df.apply(unit_normal, axis=1)
plot_df["n_x"] = normals.apply(lambda n: n[0])
plot_df["n_y"] = normals.apply(lambda n: n[1])
plot_df["n_z"] = normals.apply(lambda n: n[2])

# sagittal annotation point projected onto axial normal axis
plot_df["sag_projected_height"] = point_dot_normal(
    plot_df["pt_x"],
    plot_df["pt_y"],
    plot_df["pt_z"],
    plot_df["n_x"],
    plot_df["n_y"],
    plot_df["n_z"],
)

# axial slice plane position on same normal axis
plot_df["axial_slice_height"] = point_dot_normal(
    plot_df["ipp_x"],
    plot_df["ipp_y"],
    plot_df["ipp_z"],
    plot_df["n_x"],
    plot_df["n_y"],
    plot_df["n_z"],
)

plot_df["signed_height_error_mm"] = (
    plot_df["sag_projected_height"] - plot_df["axial_slice_height"]
)

plot_df["abs_height_error_mm"] = plot_df["signed_height_error_mm"].abs()


plt.figure(figsize=(7, 6))

sns.scatterplot(
    data=plot_df,
    x="sag_projected_height",
    y="axial_slice_height",
    hue="level",
    hue_order=LEVEL_ORDER,
    alpha=0.6,
)

vmin = min(
    plot_df["sag_projected_height"].min(),
    plot_df["axial_slice_height"].min(),
)
vmax = max(
    plot_df["sag_projected_height"].max(),
    plot_df["axial_slice_height"].max(),
)

plt.plot([vmin, vmax], [vmin, vmax], linestyle="--")

plt.title("Sagittal annotation height vs axial slice height")
plt.xlabel("Sagittal point projected onto axial slice axis")
plt.ylabel("Annotated axial slice position")
plt.grid(alpha=0.3)
plt.legend(title="Level", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
print("Overall:")
print("Mean abs plane distance (mm):", plot_df["abs_height_error_mm"].mean())
print("Median abs plane distance (mm):", plot_df["abs_height_error_mm"].median())

display(
    plot_df.groupby("condition")
    .agg(
        n=("abs_height_error_mm", "size"),
        mean_abs_plane_distance_mm=("abs_height_error_mm", "mean"),
        median_abs_plane_distance_mm=("abs_height_error_mm", "median"),
    )
    .reset_index()
)

display(
    plot_df.groupby(["condition", "level"])
    .agg(
        n=("abs_height_error_mm", "size"),
        mean_abs_plane_distance_mm=("abs_height_error_mm", "mean"),
        median_abs_plane_distance_mm=("abs_height_error_mm", "median"),
    )
    .reset_index()
)

### Axial slices matching

In [ ]:
import numpy as np
import pandas as pd

LEVEL_ORDER = ["L1/L2", "L2/L3", "L3/L4", "L4/L5", "L5/S1"]

GEOM_COLS = [
    "img_path_key",
    "ipp_x", "ipp_y", "ipp_z",
    "row_x", "row_y", "row_z",
    "col_x", "col_y", "col_z",
    "normal_x", "normal_y", "normal_z",
    "pixel_spacing_0", "pixel_spacing_1",
    "slice_rank_true", "n_slices_true",
    "img_path",
]

In [ ]:
def attach_geometry(df, all_slices_geom):
    df = df.copy()

    # avoid duplicate geometry columns from older experiments
    drop_cols = [c for c in GEOM_COLS if c in df.columns and c != "img_path_key"]
    df = df.drop(columns=drop_cols, errors="ignore")

    geom = all_slices_geom[GEOM_COLS].drop_duplicates("img_path_key").copy()

    return df.merge(
        geom,
        on="img_path_key",
        how="left",
        suffixes=("", "_geom"),
    )

In [ ]:
def select_axial_triplets_from_sagittal_points(df_split, all_slices_geom):
    df = attach_geometry(df_split, all_slices_geom)

    df["study_id"] = df["study_id"].astype(str)
    df["series_id"] = df["series_id"].astype(str)

    # Sagittal level reference point: spinal canal annotation per study + level
    sag_ref = df[
        df["plane"].eq("Sagittal")
        & df["condition"].eq("Spinal Canal Stenosis")
        & df["level"].isin(LEVEL_ORDER)
    ].copy()

    sag_points = sag_ref.apply(pixel_to_patient_3d, axis=1)

    sag_ref["pt_x"] = sag_points.apply(lambda p: p[0])
    sag_ref["pt_y"] = sag_points.apply(lambda p: p[1])
    sag_ref["pt_z"] = sag_points.apply(lambda p: p[2])

    sag_ref = (
        sag_ref[["study_id", "level", "pt_x", "pt_y", "pt_z"]]
        .dropna()
        .drop_duplicates(["study_id", "level"])
    )

    # Axial annotations to evaluate
    axial_ann = df[
        df["plane"].eq("Axial")
        & df["level"].isin(LEVEL_ORDER)
        & df["slice_rank_true"].notna()
    ].copy()

    axial_ann = axial_ann.merge(
        sag_ref,
        on=["study_id", "level"],
        how="inner",
    )

    # Full axial slice geometry, not just annotated rows
    axial_slices = all_slices_geom[
        all_slices_geom["plane"].eq("Axial")
    ].copy()

    axial_slices["study_id"] = axial_slices["study_id"].astype(str)
    axial_slices["series_id"] = axial_slices["series_id"].astype(str)
    axial_slices["slice_rank_true"] = axial_slices["slice_rank_true"].astype(int)

    selected_rows = []

    for _, row in axial_ann.iterrows():
        study_id = str(row["study_id"])
        series_id = str(row["series_id"])

        series_slices = axial_slices[
            axial_slices["study_id"].eq(study_id)
            & axial_slices["series_id"].eq(series_id)
        ].copy()

        if len(series_slices) == 0:
            continue

        point = np.array([row["pt_x"], row["pt_y"], row["pt_z"]], dtype=float)

        origins = series_slices[["ipp_x", "ipp_y", "ipp_z"]].values.astype(float)
        normals = series_slices[["normal_x", "normal_y", "normal_z"]].values.astype(float)

        normal_norms = np.linalg.norm(normals, axis=1, keepdims=True)
        normals_unit = normals / normal_norms

        distances = np.abs(np.sum((point - origins) * normals_unit, axis=1))

        nearest_idx = int(np.argmin(distances))
        center_rank = int(series_slices.iloc[nearest_idx]["slice_rank_true"])
        n_slices = int(series_slices["n_slices_true"].iloc[0])

        triplet_ranks = [
            int(np.clip(center_rank - 1, 1, n_slices)),
            int(np.clip(center_rank, 1, n_slices)),
            int(np.clip(center_rank + 1, 1, n_slices)),
        ]

        rank_to_path = dict(
            zip(
                series_slices["slice_rank_true"].astype(int),
                series_slices["img_path"],
            )
        )

        triplet_paths = [rank_to_path.get(r) for r in triplet_ranks]

        true_rank = int(row["slice_rank_true"])

        out = row.to_dict()
        out.update({
            "selector_method": "sagittal_point_to_nearest_axial_slice",
            "selected_center_rank": center_rank,
            "selected_triplet_ranks": triplet_ranks,
            "selected_img_paths": triplet_paths,
            "selected_img_path_prev": triplet_paths[0],
            "selected_img_path_center": triplet_paths[1],
            "selected_img_path_next": triplet_paths[2],
            "selected_n_slices": n_slices,
            "nearest_plane_distance_mm": float(distances[nearest_idx]),
            "rank_error": abs(center_rank - true_rank),
            "annotated_slice_covered": true_rank in triplet_ranks,
        })

        selected_rows.append(out)

    return pd.DataFrame(selected_rows)

In [ ]:
train_axial_geom_selected_df = select_axial_triplets_from_sagittal_points(
    train_df,
    all_slices_geom,
)

val_axial_geom_selected_df = select_axial_triplets_from_sagittal_points(
    val_df,
    all_slices_geom,
)

test_axial_geom_selected_df = select_axial_triplets_from_sagittal_points(
    test_df,
    all_slices_geom,
)

print("Selected rows:")
print("train:", train_axial_geom_selected_df.shape)
print("val:  ", val_axial_geom_selected_df.shape)
print("test: ", test_axial_geom_selected_df.shape)

In [ ]:
# Reuse the exact same evaluation functions as sagittal selection

train_axial_geom_selected_df = add_annotated_slice_coverage(train_axial_geom_selected_df)
val_axial_geom_selected_df = add_annotated_slice_coverage(val_axial_geom_selected_df)
test_axial_geom_selected_df = add_annotated_slice_coverage(test_axial_geom_selected_df)

train_axial_geom_selected_df = add_triplet_interval_columns(train_axial_geom_selected_df)
val_axial_geom_selected_df = add_triplet_interval_columns(val_axial_geom_selected_df)
test_axial_geom_selected_df = add_triplet_interval_columns(test_axial_geom_selected_df)

In [ ]:
report_annotated_slice_coverage(
    train_axial_geom_selected_df,
    "TRAIN AXIAL GEOMETRY"
)

report_annotated_slice_coverage(
    val_axial_geom_selected_df,
    "VALIDATION AXIAL GEOMETRY"
)

report_annotated_slice_coverage(
    test_axial_geom_selected_df,
    "TEST AXIAL GEOMETRY"
)

In [ ]:
def report_axial_geometry_selection(df, name):
    print("=" * 80)
    print(name)
    print("=" * 80)

    print("Coverage:", df["annotated_slice_covered"].mean())
    print("Mean rank error:", df["rank_error"].mean())
    print("Median rank error:", df["rank_error"].median())
    print("Mean nearest plane distance mm:", df["nearest_plane_distance_mm"].mean())
    print("Median nearest plane distance mm:", df["nearest_plane_distance_mm"].median())

    print("\nBy condition:")
    display(
        df.groupby("condition")
        .agg(
            n=("annotated_slice_covered", "size"),
            coverage=("annotated_slice_covered", "mean"),
            mean_rank_error=("rank_error", "mean"),
            median_rank_error=("rank_error", "median"),
            mean_nearest_plane_distance_mm=("nearest_plane_distance_mm", "mean"),
            median_nearest_plane_distance_mm=("nearest_plane_distance_mm", "median"),
        )
        .reset_index()
    )

    print("\nBy condition and level:")
    display(
        df.groupby(["condition", "level"])
        .agg(
            n=("annotated_slice_covered", "size"),
            coverage=("annotated_slice_covered", "mean"),
            mean_rank_error=("rank_error", "mean"),
            median_rank_error=("rank_error", "median"),
            mean_nearest_plane_distance_mm=("nearest_plane_distance_mm", "mean"),
            median_nearest_plane_distance_mm=("nearest_plane_distance_mm", "median"),
        )
        .reset_index()
    )

In [ ]:
report_axial_geometry_selection(train_axial_geom_selected_df, "TRAIN AXIAL GEOMETRY SELECTION")
report_axial_geometry_selection(val_axial_geom_selected_df, "VALIDATION AXIAL GEOMETRY SELECTION")
report_axial_geometry_selection(test_axial_geom_selected_df, "TEST AXIAL GEOMETRY SELECTION")